# Zero-shot 난이도 probe (Qwen3.5-2B / 4B)

학습 없이, 모든 모델에 **같은 prompt · 같은 이미지 · 같은 a/b/c/d 보기**를 넣고 다음 토큰의 a/b/c/d logits로 `P(correct)`를 구해 train 샘플 난이도를 매깁니다.

| 결과 | 난이도 |
|---|---|
| 2B, 4B 모두 정답 | Easy |
| 2B 실패, 4B 정답 | Medium |
| 둘 다 실패 | Hard (모델 취약 유형) |
| 2B 정답, 4B 실패 | Inconsistent (운/노이즈 가능성) |

연속 점수: `difficulty = 1 - 가중평균 P(correct)` (기본은 동일 가중 = 평균 confidence).

**Sanity check**: dev는 사람도 2:3으로 갈린 모호한 문항이므로, 정상적인 probe라면 train 대비 dev에서 accuracy ↓, P(correct) ↓, prediction entropy ↑ 가 나와야 합니다.

> **모델 형식**: `unsloth/Qwen3.5-*-GGUF`는 llama.cpp 전용 포맷이라 이 환경(transformers/unsloth)에서 logits를 뽑을 수 없습니다. 같은 가중치의 safetensors 저장소 `unsloth/Qwen3.5-2B`, `unsloth/Qwen3.5-4B`를 **bf16(비양자화)** 로 로드합니다. 8B probe를 추가하려면 `PROBES`에 한 줄만 넣으면 됩니다(작은 모델 → 큰 모델 순서).

In [1]:
import os, sys, subprocess
from pathlib import Path

LAB = Path("/opt/lab/start-lab.py").exists()            # Docker Jupyter lab 여부
REPO = Path("/workspace/scene-text-vision-repo") if LAB else Path.cwd().resolve().parents[0] if Path.cwd().name == "eda" else Path.cwd()
os.chdir(REPO); sys.path.insert(0, str(REPO / "eda"))
print("repo:", REPO, "| lab:", LAB)

# 작은 모델 → 큰 모델 순서. 값 = (HF 모델 id, 결과 폴더, 배치 크기)
# 배치 크기는 RTX 5060 Ti 16GB에서 `stv-bench`로 잰 값: prefill이 연산 병목이라 배치 2~8은 속도가 같고 그 이상은 느려짐.
#   2B: 8 → 약 5.6 img/s, peak 6.2GiB (32까지 적재 가능, 48 OOM)
#   4B: 4 → 약 3.5 img/s, peak 10.3GiB (8까지 적재 가능, 12 OOM)
PROBES = {
    "2b": ("unsloth/Qwen3.5-2B", "outputs/probe_2b", 8),
    "4b": ("unsloth/Qwen3.5-4B", "outputs/probe_4b", 4),
}
SPLITS = ["train", "dev"]
LIMIT = 0        # 0 = 전체. 먼저 200 정도로 돌려 보고 0으로 바꾸세요 (LIMIT을 바꾸면 기존 npz를 지우고 다시 실행)

repo: /workspace/scene-text-vision-repo | lab: True


## 1. 모델 다운로드 (최초 1회)
Docker lab은 HF 미러가 닿지 않아 이 셀에서만 `huggingface.co`로 직접 받습니다(2B ≈ 4.5GB, 4B ≈ 9GB). 이미 캐시에 있으면 바로 끝납니다.

In [2]:
env = dict(os.environ, HF_HUB_OFFLINE="0", HF_ENDPOINT="https://huggingface.co")
for name, (model_id, _, _) in PROBES.items():
    subprocess.run([sys.executable, "-c",
        f"from huggingface_hub import snapshot_download; print(snapshot_download({model_id!r}))"], env=env, check=True)

Fetching 16 files: 100%|██████████| 16/16 [01:21<00:00,  5.11s/it]


/root/.cache/huggingface/hub/models--unsloth--Qwen3.5-2B/snapshots/fffbc0d8711cf87df6d1f97938c5e50ef94aadc7


Fetching 17 files:  94%|█████████▍| 16/17 [02:53<00:10, 10.84s/it]
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 88, in _inner_fn
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/_snapshot_download.py", line 455, in snapshot_download
  File "/usr/local/lib/python3.12/dist-packages/tqdm/contrib/concurrent.py", line 191, in thread_map
  File "/usr/local/lib/python3.12/dist-packages/tqdm/contrib/concurrent.py", line 163, in _executor_map
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 619, in result_iterator
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 317, in _result_or_cancel
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 456, in result
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 401, in __get_result
  File "/usr/lib/python3.12/concurrent/futures/thread.py", line 58, in run
  File "/usr/

CalledProcessError: Command '['/usr/bin/python3', '-c', "from huggingface_hub import snapshot_download; print(snapshot_download('unsloth/Qwen3.5-4B'))"]' returned non-zero exit status 1.

## 2. Zero-shot 추론
모델·split마다 별도 프로세스로 실행해 끝나면 VRAM이 해제됩니다. 결과(`{split}_probs.npz`)가 이미 있으면 건너뜁니다. 전체(train 6,714 + dev 2,683) 기준 2B 약 30분, 4B 약 45분.
`--valid-ratio 0 --split train` = train.csv 전체, `--no-load-in-4bit` = bf16 비양자화, `--adapter-dir none` = zero-shot.

In [ ]:
env = dict(os.environ, HF_HUB_OFFLINE="1") if LAB else dict(os.environ)
for name, (model_id, out_dir, batch) in PROBES.items():
    for split in SPLITS:
        if (REPO / out_dir / f"{split}_probs.npz").exists():
            print(f"[skip] {name} {split}"); continue
        cmd = [sys.executable, "-m", "stv.inference", "--model-id", model_id, "--adapter-dir", "none",
               "--no-load-in-4bit", "--split", split, "--valid-ratio", "0",
               "--max-infer-samples", str(LIMIT), "--infer-batch-size", str(batch), "--output-dir", out_dir]
        print("$", " ".join(cmd)); subprocess.run(cmd, env=env, check=True)

## 3. 샘플별 결과표

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from difficulty import difficulty_table, probe_summary, tier_counts, TIER_ORDER
from common import question_type

probe_dirs = {name: out_dir for name, (_, out_dir, _) in PROBES.items()}
# 가중 점수를 쓰려면 예: weights={"2b": 0.4, "4b": 0.6}
tables = {split: difficulty_table(probe_dirs, split) for split in SPLITS}
train, dev = tables["train"], tables["dev"]
train.head()

## 4. Sanity check: train vs dev
기대 패턴: dev에서 accuracy ↓, P(correct) ↓, entropy ↑. dev 정답은 annotator 다수결이며, 동률(is_tie) 문항을 뺀 값도 같이 봅니다.

In [ ]:
summary = pd.concat({
    "train": probe_summary(train, probe_dirs),
    "dev": probe_summary(dev, probe_dirs),
    "dev (동률 제외)": probe_summary(dev[~dev.is_tie], probe_dirs),
}).round(4)
display(summary)

for name in PROBES:
    t, d = summary.loc[("train", name)], summary.loc[("dev", name)]
    ok = t.accuracy > d.accuracy and t.p_correct_mean > d.p_correct_mean and t.entropy_mean < d.entropy_mean
    print(f"{name}: {'OK  - 기대 패턴과 일치' if ok else 'CHECK - 기대 패턴과 다름'}")

In [ ]:
fig, axes = plt.subplots(1, len(PROBES), figsize=(6 * len(PROBES), 3.5), sharey=True)
for ax, name in zip(np.atleast_1d(axes), PROBES):
    bins = np.linspace(0, 1, 21)
    ax.hist(train[f"p_correct_{name}"], bins=bins, density=True, alpha=0.6, label="train")
    ax.hist(dev[f"p_correct_{name}"], bins=bins, density=True, alpha=0.6, label="dev")
    ax.set_title(f"P(correct) - {name}"); ax.set_xlabel("P(correct)"); ax.legend()
plt.tight_layout(); plt.show()

## 5. 난이도 구분 (train)

In [ ]:
display(pd.concat({"train": tier_counts(train), "dev": tier_counts(dev)}, axis=1))

names = list(PROBES)
if len(names) == 2:
    a, b = names
    print("\n정답 여부 교차표 (train)")
    display(pd.crosstab(train[f"{a}_correct"], train[f"{b}_correct"], margins=True))

train.difficulty.hist(bins=40, figsize=(7, 3)); plt.title("train difficulty = 1 - mean P(correct)"); plt.show()

## 6. 어떤 문항이 어려운가

In [ ]:
train["question_type"] = train.question.map(question_type)
by_type = train.groupby("question_type").agg(
    n=("id", "size"), difficulty=("difficulty", "mean"),
    hard_pct=("tier", lambda s: (s == "Hard").mean() * 100),
    **{f"acc_{n}": (f"{n}_correct", "mean") for n in PROBES},
).sort_values("difficulty", ascending=False).round(3)
display(by_type)

# dev: 사람 합의율이 낮을수록 모델도 어려워하는가
print("dev: consensus_ratio vs difficulty 상관 =", round(dev.consensus_ratio.corr(dev.difficulty), 3))

cols = ["id", "question", "answer"] + [f"pred_{n}" for n in PROBES] + [f"p_correct_{n}" for n in PROBES] + ["difficulty"]
print("\n가장 어려운 train 문항 20개 (두 모델 모두 확신을 갖고 틀림 → 라벨 오류 후보이기도 함)")
display(train.sort_values("difficulty", ascending=False)[cols].head(20))

## 7. 저장
`eda/tables/difficulty_{split}.csv` → 학습 시 난이도별 샘플링·가중치·오답 분석에 사용합니다.

In [ ]:
for split, table in tables.items():
    path = REPO / "eda" / "tables" / f"difficulty_{split}.csv"
    table.drop(columns=["question"]).to_csv(path, index=False)
    print("saved", path, len(table))